Generate a chart from a preprocessed `.pt` song

Runs top to bottom: setup, load model, define the generator, then set song settings and run.

In [1]:
import sys, json, warnings
import torch
from pathlib import Path
from hydra import initialize_config_dir, compose

PROJECT_ROOT = Path.cwd().parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.append(str(PROJECT_ROOT))

CONFIG_DIR = PROJECT_ROOT / "conf"
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

with initialize_config_dir(version_base=None, config_dir=str(CONFIG_DIR)):
    cfg = compose(config_name="train")

experiment_config = cfg.experiment
hyperparams = cfg.hyperparams
segment_frames = hyperparams["segment_frames"]

warnings.filterwarnings("ignore", message=".*nested tensors.*")

In [2]:
from tracers.models.transformer import TracerGlobalNpModel
from tracers.training.trainer import Trainer
from tracers.generate.converter import predictions_to_json, export_generation_debug, generate_custom_song

model = TracerGlobalNpModel(hyperparams).to(device)

# The optimizer is only here because load_checkpoint() also restores its state.
optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=hyperparams["learning_rate"],
    weight_decay=hyperparams["weight_decay"],
)

# No data loaders are needed for generation.
trainer = Trainer(
    model=model,
    optimizer=optimizer,
    train_loader=None,
    valid_loader=None,
    device=device,
    experiment_config=experiment_config,
    hyperparams=hyperparams,
)

CHECKPOINT = PROJECT_ROOT / "notebooks/checkpoints/best.pt"
trainer.load_checkpoint(CHECKPOINT)

Loaded c:\Users\joshua\vsjycode\csc173\transformers\TRACERS\notebooks\checkpoints\best.pt (epoch 39, best val loss 0.1490)


In [ ]:
# ---------- settings ----------
AUDIO_PT_NAME = "renai"
AUDIO_PT = PROJECT_ROOT / f"{AUDIO_PT_NAME}.pt"
DIFFICULTY = 5
BPM = 120            
OFFSET_MS = 0        
LANE_MAP = (0, 2, 4, 6)

TEMPERATURE = 0.4   # None = greedy
TOP_K = None
TOP_P = None
SEED = 0
SAVE_DEBUG_CSV = True

# ---------- generate ----------
generated, debug_segments = generate_custom_song(
    trainer,
    AUDIO_PT,
    difficulty=DIFFICULTY,
    segment_frames=segment_frames,
    context_frames=500,
    seed_action=0,
    seed=SEED,
    temperature=TEMPERATURE,
    top_k=TOP_K,
    top_p=TOP_P,
    collect_debug=SAVE_DEBUG_CSV,
)


# ---------- export ----------
events = predictions_to_json(
    action_predictions=generated,
    bpm=BPM,
    start_frame=0,
    offset_ms=OFFSET_MS,
    lane_map=LANE_MAP,
    valid_frames=torch.ones(len(generated), dtype=torch.bool),
)

save_to = "generated_chart_global.json"
with open(f"charts/{AUDIO_PT_NAME}/{save_to}", "w", encoding="utf-8") as f:
    json.dump(events, f, indent=4)
print(f"Saved {save_to}")

if SAVE_DEBUG_CSV:
    export_generation_debug(debug_segments, output_path=f"charts/{AUDIO_PT_NAME}/generation_debug.csv", top_k=5)

Prediction frames: 25475 (254.8 s)
Window size: 2000 | Context: 500 | Stride: 1500
Window      1-  2000 | Context:    0 | New: 2000 | Active:    1
Window   1501-  3500 | Context:  500 | New: 1500 | Active:    0
Window   3001-  5000 | Context:  500 | New: 1500 | Active:    0
Window   4501-  6500 | Context:  500 | New: 1500 | Active:    0
Window   6001-  8000 | Context:  500 | New: 1500 | Active:    0
Window   7501-  9500 | Context:  500 | New: 1500 | Active:    0
Window   9001- 11000 | Context:  500 | New: 1500 | Active:    0
Window  10501- 12500 | Context:  500 | New: 1500 | Active:    0
Window  12001- 14000 | Context:  500 | New: 1500 | Active:    0
Window  13501- 15500 | Context:  500 | New: 1500 | Active:    1
Window  15001- 17000 | Context:  500 | New: 1500 | Active:    0
